# МАНГА

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


In [1]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/"



Mounted at /content/drive


In [2]:
# =====================================================
# УСТАНОВКА ЗАВИСИМОСТЕЙ
# =====================================================

!pip install ultralytics -q
!pip install paddleocr -q
!pip install paddlepaddle -q
!pip install opencv-python-headless -q
!pip install pillow -q
!pip install easyocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7

In [3]:
# =====================================================
# ИМПОРТЫ
# =====================================================

import os
import re
import json
import zipfile
import unicodedata
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from ultralytics import YOLO
import easyocr

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


РАСПАКОВКА АРХИВА

In [4]:
zip_path = "/content/drive/MyDrive/MangaLangV.zip"

extract_path = "/content/MangaLang"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Архив распакован")

Архив распакован


ЗАГРУЗКА YOLO

In [5]:
from ultralytics import YOLO

# новая обученная модель
MODEL_PATH = "/content/drive/MyDrive/best_yolo_model_V.pt"

model = YOLO(MODEL_PATH)

print("YOLO загружена.")

YOLO загружена.


ВЫБОР ЯЗЫКА

In [6]:
# =====================================================
# НАСТРОЙКА ЯЗЫКОВ
# =====================================================

LANGUAGES = {
    "0": {
        "name": "Chinese (Simplified)",
        "code": "ch_sim",
        "folder": "China",
        "ocr": "easyocr",
        "direction": "horizontal"  # для китайского горизонтальный
    },
    "1": {
        "name": "Korean",
        "code": "ko",
        "folder": "Korea",
        "ocr": "easyocr",
        "direction": "horizontal"  # для корейского горизонтальный
    }
}

In [7]:
print("Доступные языки")

for k,v in LANGUAGES.items():
    print(k, v["name"])

lang_choice = input("Выберите язык: ").strip()

selected_lang = LANGUAGES[lang_choice]

print(selected_lang["name"])

Доступные языки
0 Chinese (Simplified)
1 Korean
Выберите язык: 1
Korean


ПОИСК ИЗОБРАЖЕНИЙ

In [8]:
base_path = "/content/MangaLang/MangaLang"

lang_folder = os.path.join(
    base_path,
    selected_lang["folder"]
)

test_images = []

for root, dirs, files in os.walk(lang_folder):

    for file in files:

        if file.lower().endswith(
            (".jpg",".jpeg",".png")
        ):

            test_images.append(
                os.path.join(root,file)
            )

print("Найдено:", len(test_images))

Найдено: 6


Детектирование пузырей и текста

In [9]:
import cv2
import os
from pathlib import Path

# ----------------------------------------
# настройки
# ----------------------------------------

IMAGE_FOLDER = "/content/images"

# минимальная уверенность

BUBBLE_CONF = 0.30
TEXT_CONF   = 0.25

# ----------------------------------------

def box_inside(box_small, box_big, margin=5):
    """
    Проверяет находится ли текст внутри пузыря.
    """

    sx1, sy1, sx2, sy2 = box_small
    bx1, by1, bx2, by2 = box_big

    return (
        sx1 >= bx1-margin and
        sy1 >= by1-margin and
        sx2 <= bx2+margin and
        sy2 <= by2+margin
    )

# ----------------------------------------

all_pages = []

# ----------------------------------------

test_images = sorted(test_images)

for image_path in test_images:

    image_name = os.path.basename(image_path)

    image = cv2.imread(image_path)

    if image is None:
        print(f"Не удалось открыть {image_path}")
        continue

    h, w = image.shape[:2]

    result = model.predict(
        image,
        conf=min(BUBBLE_CONF, TEXT_CONF),
        verbose=False
    )[0]

    bubbles=[]
    texts=[]

    # ------------------------------------

    for box in result.boxes:

        cls=int(box.cls[0])

        conf=float(box.conf[0])

        x1,y1,x2,y2=map(int,box.xyxy[0])

        if cls==0 and conf>=BUBBLE_CONF:

            bubbles.append({
                "bbox":[x1,y1,x2,y2],
                "confidence":conf
            })

        elif cls==1 and conf>=TEXT_CONF:

            texts.append({
                "bbox":[x1,y1,x2,y2],
                "confidence":conf
            })

    # ------------------------------------
    # определяем текст внутри пузыря
    # ------------------------------------

    bubble_texts=[]

    outside_texts=[]

    for t in texts:

        inside=False

        for b in bubbles:

            if box_inside(t["bbox"],b["bbox"]):

                bubble_texts.append(t)

                inside=True

                break

        if not inside:

            outside_texts.append(t)

    # ------------------------------------

    page={

        "image":image_name,

        "bubbles":bubbles,

        "texts_in_bubbles":bubble_texts,

        "texts_outside":outside_texts

    }

    all_pages.append(page)

# ----------------------------------------

print("Страниц:",len(all_pages))

print()

Страниц: 6



In [10]:
print("selected_lang =", selected_lang)
print("type =", type(selected_lang))
print("keys =", selected_lang.keys())
print("code =", selected_lang.get("code"))

selected_lang = {'name': 'Korean', 'code': 'ko', 'folder': 'Korea', 'ocr': 'easyocr', 'direction': 'horizontal'}
type = <class 'dict'>
keys = dict_keys(['name', 'code', 'folder', 'ocr', 'direction'])
code = ko


OCR

In [11]:
import os
import re
import json
import unicodedata

import cv2
import numpy as np

from PIL import Image
!pip install easyocr -q
import easyocr

In [12]:
BUBBLE_CLASS = 0
TEXT_CLASS = 1

BUBBLE_CONF = 0.30
TEXT_CONF = 0.25

In [13]:
# Функция проверки "текст внутри пузыря"
def text_inside_bubble(text_box, bubble_box):

    tx1, ty1, tx2, ty2 = text_box
    bx1, by1, bx2, by2 = bubble_box

    cx = (tx1 + tx2) / 2
    cy = (ty1 + ty2) / 2

    return (
        bx1 <= cx <= bx2 and
        by1 <= cy <= by2
    )

In [14]:
print(easyocr.__version__)

1.7.2


In [15]:
# =====================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =====================================================

def box_inside(box_small, box_big, margin=5):
    """Проверяет, находится ли текст внутри пузыря"""
    sx1, sy1, sx2, sy2 = box_small
    bx1, by1, bx2, by2 = box_big
    return (sx1 >= bx1 - margin and sy1 >= by1 - margin and
            sx2 <= bx2 + margin and sy2 <= by2 + margin)

def text_inside_bubble(text_box, bubble_box):
    """Проверяет, находится ли центр текста внутри пузыря"""
    tx1, ty1, tx2, ty2 = text_box
    bx1, by1, bx2, by2 = bubble_box
    cx = (tx1 + tx2) / 2
    cy = (ty1 + ty2) / 2
    return (bx1 <= cx <= bx2 and by1 <= cy <= by2)

def clean_text(text):
    """Базовая очистка текста"""
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def sort_text_boxes(text_items, y_threshold=20):
    """Сортировка текстов внутри пузыря: сверху вниз, слева направо"""
    if len(text_items) <= 1:
        return text_items

    text_items = sorted(text_items, key=lambda x: (x["bbox"][1], x["bbox"][0]))

    rows = []
    for item in text_items:
        x1, y1, x2, y2 = item["bbox"]
        center_y = (y1 + y2) / 2
        found = False

        for row in rows:
            if abs(center_y - row["center"]) < y_threshold:
                row["items"].append(item)
                row["center"] = np.mean([
                    (t["bbox"][1] + t["bbox"][3]) / 2
                    for t in row["items"]
                ])
                found = True
                break

        if not found:
            rows.append({"center": center_y, "items": [item]})

    rows = sorted(rows, key=lambda r: r["center"])
    result = []
    for row in rows:
        row["items"] = sorted(row["items"], key=lambda t: t["bbox"][0])
        result.extend(row["items"])

    return result

def merge_text_items(text_items):
    """Объединение текста из нескольких боксов"""
    if len(text_items) == 0:
        return ""

    text_items = sort_text_boxes(text_items)
    parts = [item["text"].strip() for item in text_items if item["text"].strip()]
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" !", "!")
    text = text.replace(" ?", "?").replace(" ;", ";").replace(" :", ":")

    return text.strip()




In [16]:
# =====================================================
# ПРЕДОБРАБОТКА ДЛЯ OCR
# =====================================================

def preprocess_variant1(img):
    """Вариант 1: CLAHE + увеличение"""
    if img is None or img.size == 0:
        return img

    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    h, w = gray.shape
    scale = 2 if max(h, w) >= 120 else 3
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    return gray

def preprocess_variant2(img):
    """Вариант 2: CLAHE + билатеральный фильтр + увеличение"""
    if img is None or img.size == 0:
        return img

    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    gray = cv2.bilateralFilter(gray, d=5, sigmaColor=35, sigmaSpace=35)

    h, w = gray.shape
    scale = 2 if max(h, w) >= 120 else 3
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    return gray

def preprocess_variant3(img):
    """Вариант 3: CLAHE + увеличение + резкость"""
    if img is None or img.size == 0:
        return img

    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    h, w = gray.shape
    scale = 2 if max(h, w) >= 120 else 3
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    blur = cv2.GaussianBlur(gray, (0, 0), 1.0)
    gray = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)

    return gray

In [17]:
# =====================================================
# ВЫБОР ЛУЧШЕГО РЕЗУЛЬТАТА OCR
# =====================================================

def score_text(text, lang_code):
    """Оценивает качество распознанного текста для выбора лучшего варианта"""
    if not text:
        return -1000

    score = 0
    score += len(text)

    # Поощряем алфавитные символы
    letters = sum(c.isalpha() for c in text)
    score += letters

    # Для китайского и корейского поощряем символы в нужных диапазонах
    for ch in text:
        if lang_code == "ch_sim":
            # Китайские иероглифы (CJK Unified Ideographs)
            if 0x4E00 <= ord(ch) <= 0x9FFF:
                score += 10
        elif lang_code == "ko":
            # Корейские слоги (Hangul)
            if 0xAC00 <= ord(ch) <= 0xD7A3:
                score += 10

    # Штраф за мусорные символы
    bad = "|[]{}\\/<>"
    for ch in bad:
        score -= text.count(ch) * 12

    # Штраф за слишком короткие слова
    if letters < 3:
        score -= 30

    return score

In [18]:
def choose_best_result(candidates, lang_code):
    """Выбирает лучший результат из нескольких вариантов OCR"""
    best_text = ""
    best_score = -999999

    for txt in candidates:
        txt = clean_text(txt)
        s = score_text(txt, lang_code)

        if s > best_score:
            best_score = s
            best_text = txt

    return best_text

In [19]:
# =====================================================
# ПОСТ-ОБРАБОТКА ДЛЯ КИТАЙСКОГО И КОРЕЙСКОГО
# =====================================================

def postprocess_chinese(text):
    """Пост-обработка для китайского текста"""
    if not text:
        return text

    # Удаляем лишние пробелы (китайский не использует пробелы между словами)
    text = re.sub(r"\s+", "", text)

    # Удаляем мусорные символы
    text = re.sub(r"[{}#;:\"\\]", "", text)

    # Нормализуем пунктуацию
    text = text.replace("，", ",").replace("。", ".").replace("！", "!")
    text = text.replace("？", "?").replace("；", ";").replace("：", ":")

    return text.strip()


In [20]:
def postprocess_korean(text):
    """
    Универсальная пост-обработка для корейского текста
    БЕЗ хардкода — только общие правила
    """
    if not text:
        return text

    # =============================================
    # 1. НОРМАЛИЗАЦИЯ ПРОБЕЛОВ
    # =============================================

    # Убираем пробелы перед пунктуацией
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Убираем множественные пробелы
    text = re.sub(r'\s+', ' ', text)

    # Склеиваем частицы со словами (в корейском они пишутся слитно)
    text = re.sub(r'([가-힣])\s+([은는이가을를])(\s|$)', r'\1\2\3', text)
    text = re.sub(r'([가-힣])\s+([이에])(\s|$)', r'\1\2\3', text)

    # =============================================
    # 2. ИСПРАВЛЕНИЕ ГЛАСНЫХ (ОБЩИЕ ПРАВИЛА)
    # =============================================

    # ㅐ → ㅏ в окончаниях (перед согласными)
    text = re.sub(r'ㅐ([ㄷㅆㅅ])(\s|$)', r'ㅏ\1\2', text)

    # ㅖ → ㅔ в окончаниях
    text = re.sub(r'ㅖ([ㄷㅆㅅㄱ])(\s|$)', r'ㅔ\1\2', text)

    # ㅓ → ㅕ в некоторых позициях
    text = re.sub(r'([ㄱ-ㅎ])ㅓ([ㄱ-ㅎ])([가-힣])', r'\1ㅕ\2\3', text)

    # =============================================
    # 3. ИСПРАВЛЕНИЕ СОГЛАСНЫХ (ОБЩИЕ ПРАВИЛА)
    # =============================================

    # ㅈ → ㅊ в окончаниях
    text = re.sub(r'ㅈ([아어여])', r'ㅊ\1', text)

    # ㄱ → ㅋ в окончаниях
    text = re.sub(r'ㄱ([아어여])(\s|$)', r'ㅋ\1\2', text)

    # =============================================
    # 4. НОРМАЛИЗАЦИЯ ПУНКТУАЦИИ
    # =============================================

    # Троеточие
    text = re.sub(r'\.\s*\.\s*\.', '...', text)
    text = re.sub(r'\.{2,}', '...', text)

    # Вопросительный и восклицательный знаки
    text = re.sub(r'\?\s*\?', '??', text)
    text = re.sub(r'\!\s*\!', '!!', text)

    # =============================================
    # 5. УДАЛЕНИЕ АРТЕФАКТОВ
    # =============================================

    # Мусорные символы
    text = re.sub(r'[{}#;:"\\]', '', text)

    # Одиночные латинские буквы
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    return text.strip()

In [21]:
# =====================================================
# НАСТРОЙКА OCR
# =====================================================

print("\n" + "=" * 60)
print("НАСТРОЙКА OCR")
print("=" * 60)

lang_code = selected_lang["code"]
print(f"Язык: {lang_code}")

# Настройка EasyOCR для китайского и корейского
lang_list = [lang_code]

# Добавляем английский как вспомогательный (может помочь с именами и заимствованиями)
if lang_code in ["ch_sim", "ko"]:
    lang_list.append("en")

print(f"Языки OCR: {lang_list}")

ocr = easyocr.Reader(
    lang_list,
    gpu=False,
    verbose=False
)
ocr_engine_type = "easyocr"

print("OCR готов")


НАСТРОЙКА OCR
Язык: ko
Языки OCR: ['ko', 'en']


OCR готов


In [22]:
# =====================================================
# ОСНОВНАЯ ФУНКЦИЯ OCR
# =====================================================

def ocr_text_box(text_img):
    """
    Распознавание текста с учетом выбранного языка
    """
    try:
        if text_img is None or text_img.size == 0:
            return ""

        # Три варианта предобработки
        variants = [
            preprocess_variant1(text_img),
            preprocess_variant2(text_img),
            preprocess_variant3(text_img)
        ]

        candidates = []

        # Настройки для EasyOCR
        kwargs = {
            "detail": 0,
            "paragraph": False,
            "rotation_info": [0, 180]  # Поддержка перевернутого текста
        }

        # Для китайского и корейского используем beamsearch
        if lang_code in ["ch_sim", "ko"]:
            kwargs.update({
                "decoder": "beamsearch",
                "beamWidth": 5
            })

        for img in variants:
            try:
                result = ocr.readtext(img, **kwargs)
                txt = " ".join(result)
                txt = clean_text(txt)
                candidates.append(txt)
            except Exception as e:
                pass

        if len(candidates) == 0:
            return ""

        # Выбираем лучший результат
        text = choose_best_result(candidates, lang_code)

        # Пост-обработка в зависимости от языка
        if lang_code == "ch_sim":
            text = postprocess_chinese(text)
        elif lang_code == "ko":
            text = postprocess_korean(text)

        return text

    except Exception as e:
        print(f"OCR error: {e}")
        return ""

In [23]:
def draw_boxes(image, detections):

    img_vis = image.copy()

    for d in detections:

        x1, y1, x2, y2 = d["bbox"]

        cv2.rectangle(
            img_vis,
            (x1, y1),
            (x2, y2),
            (0,255,0),
            2
        )

    return img_vis

In [24]:
# =====================================================
# ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ
# =====================================================

all_pages = []

for img_path in test_images:
    print("\n" + "=" * 70)
    print(os.path.basename(img_path))
    print("=" * 70)

    img = cv2.imread(img_path)
    if img is None:
        print(f"Не удалось загрузить: {img_path}")
        continue

    # Детекция YOLO
    result = model(img)[0]

    bubble_boxes = []
    text_boxes = []

    # Разделяем классы
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = float(box.conf.item())
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls == BUBBLE_CLASS and conf >= BUBBLE_CONF:
            bubble_boxes.append({"bbox": [x1, y1, x2, y2], "conf": conf})
        elif cls == TEXT_CLASS and conf >= TEXT_CONF:
            text_boxes.append({"bbox": [x1, y1, x2, y2], "conf": conf})

    print(f"Пузырей: {len(bubble_boxes)}")
    print(f"Текстовых боксов: {len(text_boxes)}")

    page_data = []
    text_id_counter = 1

    # OCR каждого пузыря
    for bubble in bubble_boxes:
        texts_inside = []

        for text in text_boxes:
            if not text_inside_bubble(text["bbox"], bubble["bbox"]):
                continue

            tx1, ty1, tx2, ty2 = text["bbox"]
            pad = 3
            tx1 = max(0, tx1 - pad)
            ty1 = max(0, ty1 - pad)
            tx2 = min(img.shape[1], tx2 + pad)
            ty2 = min(img.shape[0], ty2 + pad)

            crop = img[ty1:ty2, tx1:tx2]
            recognized = ocr_text_box(crop)

            if recognized == "":
                continue

            if len(recognized) == 1 and recognized not in ["?", "!", ".", "…"]:
                continue

            texts_inside.append({
                "bbox": text["bbox"],
                "text": recognized,
                "text_id": text_id_counter
            })
            text_id_counter += 1

        # Сортировка
        texts_inside = sort_text_boxes(texts_inside)

        # Удаление дублей
        unique = []
        seen = set()
        for item in texts_inside:
            txt = item["text"]
            if txt in seen:
                continue
            seen.add(txt)
            unique.append(item)
        texts_inside = unique

        # Объединение текста
        full_text = merge_text_items(texts_inside)

        page_data.append({
            "bbox": bubble["bbox"],
            "texts": texts_inside,
            "source_text": full_text,
            "confidence": bubble["conf"]
        })

    # Вывод OCR
    for i, bubble in enumerate(page_data):
        print()
        print(f"Пузырь {i + 1}")
        print(bubble["source_text"])
        for text_item in bubble["texts"]:
            print(f"  ID {text_item['text_id']}: {text_item['text']}")

    # Сохраняем страницу
    all_pages.append({
        "image": os.path.basename(img_path),
        "texts": page_data
    })


01.jpg
Пузырей: 2
Текстовых боксов: 4


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Пузырь 1
지각생이라고는 생각도 못햇다.
  ID 1: 지각생이라고는 생각도 못햇다.

Пузырь 2
인간적으로 너무 당당하잡아.
  ID 2: 인간적으로 너무 당당하잡아.

02.jpg
Пузырей: 1
Текстовых боксов: 3

Пузырь 1
가만, 년 처음 보는 얼굴 인데.
  ID 1: 가만, 년 처음 보는 얼굴 인데.

03.jpg
Пузырей: 2
Текстовых боксов: 2

Пузырь 1
에? 선생님도 처음 본 거예요?
  ID 1: 에? 선생님도 처음 본 거예요?

Пузырь 2
그래? 너도 처음 밖단 말이지 벼
  ID 2: 그래? 너도 처음 밖단 말이지 벼

04.jpg
Пузырей: 2
Текстовых боксов: 2


/usr/local/lib/python3.12/dist-packages/easyocr/utils.py:221: RuntimeWarning: overflow encountered in scalar add
  curr.entries[labeling].prTotal += prBlank + prNonBlank
/usr/local/lib/python3.12/dist-packages/easyocr/utils.py:248: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prNonBlank += prNonBlank
/usr/local/lib/python3.12/dist-packages/easyocr/utils.py:249: RuntimeWarning: overflow encountered in scalar add
  curr.entries[newLabeling].prTotal += prNonBlank
/usr/local/lib/python3.12/dist-packages/easyocr/utils.py:219: RuntimeWarning: overflow encountered in scalar add
  curr.entries[labeling].prNonBlank += prNonBlank



Пузырь 1
에드락으로 끝난걸 다행으로 알아라. 그렇다면 범인은 신우다? 고딩담껑 우의한이나?
  ID 1: 에드락으로 끝난걸 다행으로 알아라. 그렇다면 범인은 신우다? 고딩담껑 우의한이나?

Пузырь 2
그래도 좀 말살애주끼... 음음...
  ID 2: 그래도 좀 말살애주끼... 음음...

05.jpg
Пузырей: 1
Текстовых боксов: 1

Пузырь 1
그렇게 살벌하게 쳐다보면 나도 도망치고 싶거문
  ID 1: 그렇게 살벌하게 쳐다보면 나도 도망치고 싶거문

06.jpg
Пузырей: 2
Текстовых боксов: 4

Пузырь 1
거화. 무섭다 니까. 수연이 살아있올 때까진 안 그랫는데.
  ID 1: 거화. 무섭다 니까. 수연이 살아있올 때까진 안 그랫는데.

Пузырь 2
생각이 바뀌없어.
  ID 2: 생각이 바뀌없어.


In [25]:
import json

with open(
    "ocr_result.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_pages,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Готово.")
print("Страниц:", len(all_pages))

Готово.
Страниц: 6


**ПЕРЕВОД**

In [26]:
# =====================================================
# ШАГ 1: УСТАНОВКА ЗАВИСИМОСТЕЙ
# =====================================================

!pip install deep-translator -q
!pip install googletrans==4.0.0-rc1 -q  # Альтернатива, если deep-translator не работает

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpx2 2.4.0 requires idna>=3.18, but you have idna 2.10 which is incompatible.
gradio-client 2.5.0 requires httpx>=0.24.1, but you have httpx 0.13.3 which is 

In [27]:
# =====================================================
# ШАГ 2: ИМПОРТЫ
# =====================================================

import json
import time
import re

from deep_translator import GoogleTranslator

# Альтернатива: если deep_translator не работает
# from googletrans import Translator
# translator = Translator()

In [28]:
# =====================================================
# ШАГ 3: НАСТРОЙКА ЯЗЫКА ПЕРЕВОДА
# =====================================================

TARGET_LANGUAGE = "ru"  # Русский

In [29]:
# =====================================================
# ШАГ 4: ЗАГРУЗКА OCR JSON
# =====================================================

OCR_JSON = "ocr_result.json"

with open(OCR_JSON, "r", encoding="utf-8") as f:
    pages = json.load(f)

print(f" Загружено страниц: {len(pages)}")

 Загружено страниц: 6


In [30]:
# =====================================================
# ШАГ 5: ПОСТ-ОБРАБОТКА ДЛЯ КОРЕЙСКОГО (ПЕРЕД ПЕРЕВОДОМ)
# =====================================================
def preprocess_korean_for_translation(text):
    """
    Универсальная подготовка корейского текста к переводу
    Без хардкода — только общие правила
    """
    if not text:
        return text

    # 1. Исправление гласных (общие правила)
    text = re.sub(r'ㅐ([ㄷㅆㅅㄴㄹ])(\s|$)', r'ㅏ\1\2', text)
    text = re.sub(r'ㅖ([ㄷㅆㅅㄱㄴㄹ])(\s|$)', r'ㅔ\1\2', text)

    # 2. Исправление согласных (общие правила)
    text = re.sub(r'ㅈ([아어여])(\s|$)', r'ㅊ\1\2', text)
    text = re.sub(r'ㄱ([아어여])(\s|$)', r'ㅋ\1\2', text)

    # 3. Склейка частиц
    text = re.sub(r'([가-힣])\s+([은는이가을를])(\s|$)', r'\1\2\3', text)

    # 4. Нормализация пробелов
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [31]:
def preprocess_chinese_for_translation(text):
    """
    Подготовка китайского текста к переводу (удаление лишних пробелов)
    """
    if not text:
        return text

    # В китайском нет пробелов между словами
    text = re.sub(r'\s+', '', text)

    return text

In [32]:
# =====================================================
# ШАГ 6: СОЗДАНИЕ ПЕРЕВОДЧИКА
# =====================================================

# Определяем язык источника по первому тексту
# или используем auto (Google Translate определит сам)

source_lang = "auto"  # Автоопределение

translator = GoogleTranslator(
    source=source_lang,
    target=TARGET_LANGUAGE
)

print(f" Язык перевода: {TARGET_LANGUAGE}")
print(f" Определение языка: {source_lang}")
print("=" * 60)

 Язык перевода: ru
 Определение языка: auto


In [33]:
# =====================================================
# ШАГ 7: ПЕРЕВОД
# =====================================================

print("\n" + "=" * 60)
print("НАЧАЛО ПЕРЕВОДА")
print("=" * 60)

translated_count = 0
error_count = 0

for page_idx, page in enumerate(pages, 1):
    print()
    print("=" * 60)
    print(f" {page_idx}/{len(pages)}: {page['image']}")
    print("=" * 60)

    for bubble_idx, bubble in enumerate(page["texts"], 1):
        src = bubble["source_text"].strip()

        if src == "":
            bubble["translated_text"] = ""
            continue

        # =============================================
        # ПРЕДОБРАБОТКА ТЕКСТА ПЕРЕД ПЕРЕВОДОМ
        # =============================================

        # Определяем язык по содержимому (упрощенно)
        # Можно пропустить, если используете auto
        text_for_translation = src

        # Если текст содержит корейские слоги
        if re.search(r'[가-힣]', src):
            text_for_translation = preprocess_korean_for_translation(src)

        # Если текст содержит китайские иероглифы
        elif re.search(r'[\u4e00-\u9fff]', src):
            text_for_translation = preprocess_chinese_for_translation(src)

        # =============================================
        # ПЕРЕВОД
        # =============================================

        try:
            tr = translator.translate(text_for_translation)

            # Если перевод пустой или слишком короткий, пробуем еще раз
            if not tr or len(tr) < 2:
                # Небольшая задержка перед повторной попыткой
                time.sleep(0.5)
                tr = translator.translate(text_for_translation)

            bubble["translated_text"] = tr
            translated_count += 1

        except Exception as e:
            print(f"   Ошибка перевода (пузырь {bubble_idx}): {e}")
            bubble["translated_text"] = ""
            error_count += 1

        # =============================================
        # ВЫВОД РЕЗУЛЬТАТА
        # =============================================

        print()
        print(f"  Пузырь {bubble_idx}:")
        print(f"    SOURCE: {src}")
        if text_for_translation != src:
            print(f"    PREPROCESSED: {text_for_translation}")
        print(f"    TRANSLATED: {tr if tr else ' ОШИБКА'}")

        # Задержка между запросами (чтобы не заблокировали)
        time.sleep(0.3)


НАЧАЛО ПЕРЕВОДА

 1/6: 01.jpg

  Пузырь 1:
    SOURCE: 지각생이라고는 생각도 못햇다.
    TRANSLATED: Я никогда не думал, что я опоздал на учебу.

  Пузырь 2:
    SOURCE: 인간적으로 너무 당당하잡아.
    TRANSLATED: Он так горд, как человек.

 2/6: 02.jpg

  Пузырь 1:
    SOURCE: 가만, 년 처음 보는 얼굴 인데.
    TRANSLATED: Подождите, я впервые вижу это лицо.

 3/6: 03.jpg

  Пузырь 1:
    SOURCE: 에? 선생님도 처음 본 거예요?
    TRANSLATED: к? Вы тоже впервые видите вас, учитель?

  Пузырь 2:
    SOURCE: 그래? 너도 처음 밖단 말이지 벼
    TRANSLATED: хорошо? Ты тоже первый раз выходишь на улицу.

 4/6: 04.jpg

  Пузырь 1:
    SOURCE: 에드락으로 끝난걸 다행으로 알아라. 그렇다면 범인은 신우다? 고딩담껑 우의한이나?
    TRANSLATED: Слава богу, все закончилось Эдлоком. Тогда виновником является Шинву? Вы старшеклассник?

  Пузырь 2:
    SOURCE: 그래도 좀 말살애주끼... 음음...
    TRANSLATED: И все же я вам немного расскажу... ммм...

 5/6: 05.jpg

  Пузырь 1:
    SOURCE: 그렇게 살벌하게 쳐다보면 나도 도망치고 싶거문
    TRANSLATED: Если ты смотришь на меня так жестоко, я тоже хочу убежать.

 6/6: 06.jpg

  Пузы

In [34]:
# Сохранение
OUTPUT_JSON = "translated_result.json"

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pages,
        f,
        ensure_ascii=False,
        indent=4
    )

print()
print("Сохранено:", OUTPUT_JSON)


Сохранено: translated_result.json


**Отрисовка перевода**

In [35]:
# =====================================================
# ЯЧЕЙКА 1
# Импорты и подготовка проекта
# =====================================================

import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# =====================================================
# Путь к шрифту
# =====================================================

FONT_PATH = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"

if not os.path.exists(FONT_PATH):
    # Для Colab можно скачать шрифт
    !wget -q https://github.com/googlefonts/noto-cjk/raw/main/Sans/SubsetOTF/NotoSansCJK-Regular.ttc -O /content/NotoSansCJK-Regular.ttc
    FONT_PATH = "/content/NotoSansCJK-Regular.ttc"

print("Шрифт:", FONT_PATH)

# =====================================================
# Загружаем translated_result.json
# =====================================================

JSON_PATH = "translated_result.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    pages = json.load(f)

print(f"JSON загружен. Страниц: {len(pages)}")

# =====================================================
# Создаем словарь путей к изображениям
# =====================================================

image_dict = {}
for path in test_images:
    image_dict[os.path.basename(path)] = path

print(f"Изображений: {len(image_dict)}")

# Проверка соответствия
missing = [page["image"] for page in pages if page["image"] not in image_dict]
if missing:
    print("Не найдены:", missing)
else:
    print("Все изображения найдены.")

Шрифт: /usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf
JSON загружен. Страниц: 6
Изображений: 6
Все изображения найдены.


In [36]:
# =====================================================
# ЯЧЕЙКА 2
# Работа с областями текста
# =====================================================

def get_merged_text_region(bubble):
    """
    Возвращает объединенную область всех текстовых боксов БЕЗ расширения
    (для отображения и рисования)
    """
    if len(bubble["texts"]) == 0:
        return bubble["bbox"]

    x1 = min(t["bbox"][0] for t in bubble["texts"])
    y1 = min(t["bbox"][1] for t in bubble["texts"])
    x2 = max(t["bbox"][2] for t in bubble["texts"])
    y2 = max(t["bbox"][3] for t in bubble["texts"])

    return [int(x1), int(y1), int(x2), int(y2)]

def get_text_region_with_padding(bubble, padding_ratio=0.08, expand_ratio=0.25):
    """
    Возвращает расширенную область для лучшего размещения текста
    (используется только для рисования, если нужно)
    """
    bx1, by1, bx2, by2 = bubble["bbox"]

    if len(bubble["texts"]) == 0:
        return [bx1, by1, bx2, by2]

    # Объединяем все текстовые боксы
    x1 = min(t["bbox"][0] for t in bubble["texts"])
    y1 = min(t["bbox"][1] for t in bubble["texts"])
    x2 = max(t["bbox"][2] for t in bubble["texts"])
    y2 = max(t["bbox"][3] for t in bubble["texts"])

    w = x2 - x1
    h = y2 - y1

    # Добавляем отступы
    padding = max(6, int(min(w, h) * padding_ratio))
    x1 -= padding
    y1 -= padding
    x2 += padding
    y2 += padding

    # Расширяем область
    dx = int(w * expand_ratio)
    dy = int(h * expand_ratio)
    x1 -= dx
    y1 -= dy
    x2 += dx
    y2 += dy

    # Ограничиваем границами пузыря
    x1 = max(x1, bx1)
    y1 = max(y1, by1)
    x2 = min(x2, bx2)
    y2 = min(y2, by2)

    return [int(x1), int(y1), int(x2), int(y2)]

def expand_bbox(bbox, image_shape, margin=8):
    """Расширяет область для очистки"""
    x1, y1, x2, y2 = bbox
    h, w = image_shape[:2]

    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w - 1, x2 + margin)
    y2 = min(h - 1, y2 + margin)

    return [int(x1), int(y1), int(x2), int(y2)]

def estimate_background_color(image, bbox, border=3):
    """Определяет цвет фона по краям области"""
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w - 1, x2)
    y2 = min(h - 1, y2)

    pixels = []

    # Верх
    if y1 + border < y2:
        pixels.append(image[y1:y1+border, x1:x2])

    # Низ
    if y2 - border > y1:
        pixels.append(image[y2-border:y2, x1:x2])

    # Лево
    if x1 + border < x2:
        pixels.append(image[y1:y2, x1:x1+border])

    # Право
    if x2 - border > x1:
        pixels.append(image[y1:y2, x2-border:x2])

    pixels = [p.reshape(-1, 3) for p in pixels if p.size > 0]

    if len(pixels) == 0:
        return (255, 255, 255)

    pixels = np.concatenate(pixels, axis=0)
    color = np.median(pixels, axis=0)

    return tuple(map(int, color))

def clear_text_regions(image, page):
    """
    Очищает области текста внутри пузырей
    Использует ТУ ЖЕ область, что и для рисования
    """
    img = image.copy()

    for bubble in page["texts"]:
        # Используем ОБЪЕДИНЕННУЮ область (без расширения)
        region = get_merged_text_region(bubble)
        clear_region = expand_bbox(region, img.shape, margin=4)

        # Определяем цвет фона
        color = estimate_background_color(img, clear_region)

        # Заливаем область
        x1, y1, x2, y2 = clear_region
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness=-1)

    return img

print("Ячейка 2 готова.")

Ячейка 2 готова.


In [37]:
# =====================================================
# ЯЧЕЙКА 3
# Движок рисования текста с улучшенным размещением
# =====================================================

def split_text_to_lines(text, draw, font, max_width):
    """Разбивает текст на строки по ширине"""
    words = text.split()
    if len(words) == 0:
        return []

    lines = []
    current = words[0]

    for word in words[1:]:
        candidate = current + " " + word
        bbox = draw.textbbox((0, 0), candidate, font=font)
        width = bbox[2] - bbox[0]

        if width <= max_width:
            current = candidate
        else:
            lines.append(current)
            current = word

    lines.append(current)
    return lines

def fit_font_size(text, draw, font_path, box_width, box_height, min_size=4, max_size=36):
    """
    Подбирает оптимальный размер шрифта с агрессивным уменьшением
    """
    best_font = None
    best_lines = None

    # Пробуем размеры от максимального к минимальному
    for size in range(max_size, min_size - 1, -1):
        try:
            font = ImageFont.truetype(font_path, size)
        except:
            font = ImageFont.load_default()

        lines = split_text_to_lines(text, draw, font, box_width - 4)

        # Проверяем ширину каждой строки
        fits_width = True
        for line in lines:
            bbox = draw.textbbox((0, 0), line, font=font)
            if bbox[2] - bbox[0] > box_width - 4:
                fits_width = False
                break

        if not fits_width:
            continue

        # Проверяем высоту
        line_bbox = draw.textbbox((0, 0), "Ag", font=font)
        line_height = line_bbox[3] - line_bbox[1]
        total_height = line_height * len(lines)

        if total_height <= box_height - 4:
            best_font = font
            best_lines = lines
            break

    # Если ничего не подошло - берем минимальный размер и обрезаем
    if best_font is None:
        try:
            best_font = ImageFont.truetype(font_path, min_size)
        except:
            best_font = ImageFont.load_default()

        best_lines = split_text_to_lines(text, draw, best_font, box_width - 4)

        # Обрезаем строки, если не помещаются
        line_bbox = draw.textbbox((0, 0), "Ag", font=best_font)
        line_height = line_bbox[3] - line_bbox[1]
        total_height = line_height * len(best_lines)

        while total_height > box_height - 4 and len(best_lines) > 1:
            best_lines = best_lines[:-1]
            total_height = line_height * len(best_lines)

        # Если все еще не помещается - обрезаем последнюю строку
        if total_height > box_height - 4 and len(best_lines) > 0:
            last_line = best_lines[-1]
            while len(last_line) > 1:
                last_line = last_line[:-1]
                bbox = draw.textbbox((0, 0), last_line + "...", font=best_font)
                if bbox[2] - bbox[0] <= box_width - 4:
                    best_lines[-1] = last_line + "..."
                    break

    return best_font, best_lines

def get_text_color(background_color):
    """
    Определяет цвет текста на основе яркости фона
    """
    brightness = 0.299 * background_color[0] + 0.587 * background_color[1] + 0.114 * background_color[2]

    if brightness > 128:
        return (0, 0, 0)      # Черный на светлом фоне
    else:
        return (255, 255, 255) # Белый на темном фоне

def estimate_background_color_clean(image, bbox, border=5):
    """
    Определяет цвет фона ТОЛЬКО по краям области (игнорируя текст в центре)
    """
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w - 1, x2)
    y2 = min(h - 1, y2)

    if x2 - x1 < 20 or y2 - y1 < 20:
        return (255, 255, 255)

    pixels = []

    # Размер области для анализа
    border = min(border, (y2 - y1) // 5, (x2 - x1) // 5)
    border = max(border, 3)

    # =============================================
    # Берем пиксели только с КРАЕВ
    # =============================================

    # Верхняя полоса
    if y1 + border < y2:
        roi = image[y1:y1+border, x1:x2]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Нижняя полоса
    if y2 - border > y1:
        roi = image[y2-border:y2, x1:x2]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Левая полоса
    if x1 + border < x2:
        roi = image[y1:y2, x1:x1+border]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Правая полоса
    if x2 - border > x1:
        roi = image[y1:y2, x2-border:x2]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # =============================================
    # Добавляем УГЛЫ (там точно нет текста)
    # =============================================

    # Левый верхний угол
    if y1 + border < y2 and x1 + border < x2:
        roi = image[y1:y1+border, x1:x1+border]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Правый верхний угол
    if y1 + border < y2 and x2 - border > x1:
        roi = image[y1:y1+border, x2-border:x2]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Левый нижний угол
    if y2 - border > y1 and x1 + border < x2:
        roi = image[y2-border:y2, x1:x1+border]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    # Правый нижний угол
    if y2 - border > y1 and x2 - border > x1:
        roi = image[y2-border:y2, x2-border:x2]
        if roi.size > 0:
            pixels.append(roi.reshape(-1, 3))

    if len(pixels) == 0:
        return (255, 255, 255)

    all_pixels = np.concatenate(pixels, axis=0)

    if all_pixels.shape[0] == 0:
        return (255, 255, 255)

    # =============================================
    # Убираем выбросы (слишком темные/светлые пиксели)
    # =============================================

    brightness = 0.299 * all_pixels[:, 0] + 0.587 * all_pixels[:, 1] + 0.114 * all_pixels[:, 2]

    low = np.percentile(brightness, 10)
    high = np.percentile(brightness, 90)

    mask = (brightness >= low) & (brightness <= high)
    if np.any(mask):
        filtered_pixels = all_pixels[mask]
    else:
        filtered_pixels = all_pixels

    # Используем медиану
    color = np.median(filtered_pixels, axis=0)

    return tuple(map(int, color))

def put_text_in_bbox_improved(draw, text, bbox, font_path, bg_color=(255, 255, 255)):
    """
    Рисует текст в заданной области с автоматическим подбором цвета
    """
    if text.strip() == "":
        return

    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1

    if width < 10 or height < 10:
        return

    # Подбираем шрифт
    font, lines = fit_font_size(text, draw, font_path, width, height)

    # Определяем цвет текста
    text_color = get_text_color(bg_color)

    # Вычисляем высоту строки
    line_bbox = draw.textbbox((0, 0), "Ag", font=font)
    line_height = line_bbox[3] - line_bbox[1]

    total_height = line_height * len(lines)

    # Центрируем по вертикали
    y = y1 + (height - total_height) / 2

    # Определяем цвет обводки (контрастный к тексту)
    if text_color == (0, 0, 0):
        outline_color = (255, 255, 255)
    else:
        outline_color = (0, 0, 0)

    for line in lines:
        bbox_line = draw.textbbox((0, 0), line, font=font)
        line_width = bbox_line[2] - bbox_line[0]
        x = x1 + (width - line_width) / 2

        # Рисуем обводку (смещение на 1 пиксель)
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            draw.text((x + dx, y + dy), line, fill=outline_color, font=font)

        # Рисуем основной текст
        draw.text((x, y), line, fill=text_color, font=font)

        y += line_height

def clear_text_regions_improved(image, page):
    """
    Очищает области текста с улучшенным определением фона
    """
    img = image.copy()

    for bubble in page["texts"]:
        region = get_merged_text_region(bubble)
        clear_region = expand_bbox(region, img.shape, margin=4)

        # Определяем цвет фона (только по краям)
        color = estimate_background_color_clean(img, clear_region)

        # Заливаем область
        x1, y1, x2, y2 = clear_region
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness=-1)

    return img

print("Ячейка 3 готова.")

Ячейка 3 готова.


In [38]:
# =====================================================
# ЯЧЕЙКА 4
# Построение страницы с улучшенной отрисовкой
# =====================================================

def draw_translation_page_improved(image, page):
    """
    Создает изображение с переводом (улучшенная версия)
    """
    # Очищаем оригинальный текст с улучшенным определением фона
    img = clear_text_regions_improved(image, page)

    # OpenCV -> PIL
    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)

    # Рисуем перевод
    for bubble in page["texts"]:
        text = bubble.get("edited_text", "").strip()
        if text == "":
            text = bubble.get("translated_text", "").strip()

        if text == "":
            continue

        # Получаем область текста
        region = get_merged_text_region(bubble)

        # Определяем цвет фона для этой области
        bg_color = estimate_background_color_clean(img, region)

        # Рисуем текст с автоматическим подбором цвета
        put_text_in_bbox_improved(draw, text, region, FONT_PATH, bg_color)

    # PIL -> OpenCV
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

print("Ячейка 4 готова.")

Ячейка 4 готова.


In [39]:
# =====================================================
# ФУНКЦИЯ ОТРИСОВКИ СТРАНИЦЫ С ВОЗВРАТОМ КАРТИНКИ
# =====================================================

def display_and_get_translated_image(img, page):
    """
    Отображает страницу с оригиналом и переводом (с боксами и ID),
    возвращает чистую картинку с переводом (без боксов и ID)
    """
    # Копии для отрисовки
    img_original = img.copy()

    # Создаем перевод с улучшенной отрисовкой
    img_translated = draw_translation_page_improved(img, page)

    # Сохраняем чистую картинку с переводом (без боксов и ID)
    img_clean_translated = img_translated.copy()

    # =================================================
    # Рисуем боксы на ОБЕИХ картинках (для отображения)
    # =================================================

    for bubble in page["texts"]:
        # ---- Зеленый бокс пузыря ----
        bx1, by1, bx2, by2 = bubble["bbox"]

        for img_draw in [img_original, img_translated]:
            cv2.rectangle(
                img_draw,
                (bx1, by1),
                (bx2, by2),
                (0, 255, 0),  # Зеленый
                2
            )

        # ---- Красный бокс текста (объединенная область) ----
        region = get_merged_text_region(bubble)
        tx1, ty1, tx2, ty2 = region

        for img_draw in [img_original, img_translated]:
            cv2.rectangle(
                img_draw,
                (tx1, ty1),
                (tx2, ty2),
                (0, 0, 255),  # Красный
                2
            )

        # ---- Рисуем ID на правой картинке ----
        for text_item in bubble["texts"]:
            text_id = text_item.get("text_id", 0)
            if text_id > 0:
                tx1, ty1, tx2, ty2 = text_item["bbox"]

                # Вычисляем размер бокса
                box_width = tx2 - tx1
                box_height = ty2 - ty1

                # Размер шрифта пропорционален размеру бокса
                font_scale = min(box_width, box_height) / 25
                font_scale = max(0.4, min(1.2, font_scale))
                thickness = max(1, int(font_scale * 2))

                # Позиция ID (над боксом, с отступом)
                id_x = tx1
                id_y = ty1 - 10

                # Если ID выходит за верхний край картинки
                if id_y < 10:
                    id_y = ty2 + 15  # Рисуем под боксом

                # Черная обводка
                cv2.putText(
                    img_translated,
                    f"ID:{text_id}",
                    (id_x, id_y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    (0, 0, 0),
                    thickness + 2
                )

                # Основной текст ID (синий)
                cv2.putText(
                    img_translated,
                    f"ID:{text_id}",
                    (id_x, id_y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    (255, 0, 0),
                    thickness
                )

    # =================================================
    # Отображаем
    # =================================================

    plt.figure(figsize=(18, 9))
    fig = plt.figure(figsize=(18, 9))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
    plt.title(f"{os.path.basename(page['image'])}  ORIGINAL\n(Зеленый = пузырь, Красный = текст)", fontsize=12)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(img_translated, cv2.COLOR_BGR2RGB))
    plt.title(f"{os.path.basename(page['image'])}  TRANSLATED\n(Зеленый = пузырь, Красный = текст, Синий = ID)", fontsize=12)
    plt.axis("off")

    plt.tight_layout()
    #plt.show()

    # Возвращаем чистую картинку с переводом (без боксов и ID)
    return img_clean_translated, fig

In [40]:
# =====================================================
# СОЗДАНИЕ ПАПКИ ДЛЯ СОХРАНЕНИЯ
# =====================================================

# Создаем папку для сохранения переведенных изображений
lang_name = selected_lang["name"].lower()
output_dir = f"/content/translated_img_{lang_name}"
os.makedirs(output_dir, exist_ok=True)
print(f" Папка для сохранения: {output_dir}")

 Папка для сохранения: /content/translated_img_korean


In [41]:
# XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
import os
import json
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# =====================================================
# ГЛОБАЛЬНОЕ СОСТОЯНИЕ
# =====================================================

state = {
    'img': None,
    'page': None,
    'img_clean': None,
    'page_index': 0,
    'total_pages': 0,
    'image_name': '',
    'output_dir': '',
    'pages': None,
    'image_dict': None,
    'saved_count': 0,
    'stop': False,
    'current_fig': None,
}

# =====================================================
# ФУНКЦИИ ПОИСКА И СОХРАНЕНИЯ
# =====================================================

def find_text_by_id(page, text_id):
    for bubble in page["texts"]:
        for text_box in bubble["texts"]:
            if text_box.get("text_id") == text_id:
                return bubble, text_box
    return None, None

def save_current_page():
    image_name = state['image_name']
    img = state['img_clean']
    output_path = os.path.join(state['output_dir'], image_name)
    cv2.imwrite(output_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    state['saved_count'] += 1
    print(f" JPEG сохранён: {image_name}")

def save_json():
    json_path = os.path.join(state['output_dir'], "translated_result.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(state['pages'], f, ensure_ascii=False, indent=4)
    print()
    print("=" * 60)
    print("JSON сохранён:")
    print(json_path)
    print("=" * 60)

# =====================================================
# ФУНКЦИЯ ОТОБРАЖЕНИЯ КАРТИНКИ С БОКСАМИ И ID
# =====================================================

def get_translated_image_with_boxes(image, page):
    """Возвращает картинку с переводом, боксами и ID (НЕ ПОКАЗЫВАЕТ её)"""
    img_clean_translated, fig = display_and_get_translated_image(image, page)
    state['img_clean'] = img_clean_translated
    state['current_fig'] = fig
    return img_clean_translated, fig

def show_images_from_fig(fig):
    """Показывает уже созданную фигуру"""
    display(fig)
    plt.close(fig)

# =====================================================
# ФУНКЦИЯ ПОКАЗА СТРАНИЦЫ
# =====================================================

def show_page():
    """Показывает страницу с оригиналом и переводом"""

    if state['stop']:
        return

    clear_output(wait=True)

    image = state['img']
    page = state['page']
    page_index = state['page_index']
    total_pages = state['total_pages']
    image_name = state['image_name']

    print("=" * 60)
    print(f" СТРАНИЦА {page_index + 1}/{total_pages}: {image_name}")
    print("=" * 60)

    # Получаем картинку с боксами и ID
    img_clean_translated, fig = get_translated_image_with_boxes(image, page)
    state['img_clean'] = img_clean_translated

    # Показываем фигуру
    show_images_from_fig(fig)

    # Показываем меню
    show_main_menu()

# =====================================================
# ФУНКЦИЯ ГЛАВНОГО МЕНЮ
# =====================================================

def show_main_menu():
    """Показывает главное меню"""

    if state['stop']:
        return

    print("\n" + "-" * 40)
    print(" Выберите действие:")
    print("-" * 40)

    btn_edit = widgets.Button(
        description=" Редактировать",
        button_style='primary',
        layout=widgets.Layout(width='200px')
    )

    btn_save = widgets.Button(
        description=" Сохранить и далее",
        button_style='success',
        layout=widgets.Layout(width='200px')
    )

    btn_stop = widgets.Button(
        description=" Закончить",
        button_style='danger',
        layout=widgets.Layout(width='200px')
    )

    output = widgets.Output()

    def on_edit(b):
        if not state['stop']:
            show_id_selection()

    def on_save(b):
        if state['stop']:
            return
        save_current_page()

        # Переходим к следующей странице
        state['page_index'] += 1
        if state['page_index'] < state['total_pages']:
            page = state['pages'][state['page_index']]
            image_name = page["image"]

            if image_name in state['image_dict']:
                img = cv2.imread(state['image_dict'][image_name])
                if img is not None:
                    state['img'] = img
                    state['page'] = page
                    state['image_name'] = image_name
                    show_page()
                    return

        # Все страницы обработаны
        save_json()
        print("\n" + "=" * 60)
        print(" ВСЕ СТРАНИЦЫ ОБРАБОТАНЫ!")
        print("=" * 60)
        state['stop'] = True

    def on_stop(b):
        if state['stop']:
            return
        save_current_page()
        save_json()
        state['stop'] = True
        clear_output(wait=True)
        print("\n" + "=" * 60)
        print(" РАБОТА ЗАВЕРШЕНА")
        print("=" * 60)
        print(f" Сохранено страниц: {state['saved_count']}")
        print("=" * 60)
        display(widgets.HTML("<i>Редактор остановлен. Закройте эту ячейку или перезапустите для новой сессии.</i>"))

    btn_edit.on_click(on_edit)
    btn_save.on_click(on_save)
    btn_stop.on_click(on_stop)

    display(widgets.HBox([btn_edit, btn_save, btn_stop]), output)

# =====================================================
# ФУНКЦИЯ ПОКАЗА ID ДЛЯ РЕДАКТИРОВАНИЯ
# =====================================================

def show_id_selection():
    """Показывает картинку, список ID и кнопки для выбора"""

    if state['stop']:
        return

    clear_output(wait=True)

    page_index = state['page_index']
    total_pages = state['total_pages']
    image_name = state['image_name']

    print("=" * 60)
    print(f" СТРАНИЦА {page_index + 1}/{total_pages}: {image_name} (РЕДАКТИРОВАНИЕ)")
    print("=" * 60)

    # Получаем и показываем картинку с боксами и ID
    img_clean_translated, fig = get_translated_image_with_boxes(state['img'], state['page'])
    state['img_clean'] = img_clean_translated
    show_images_from_fig(fig)

    print("\n" + "=" * 60)
    print(" ДОСТУПНЫЕ ID ТЕКСТОВ")
    print("=" * 60)

    id_list = []
    for bubble in state['page']["texts"]:
        for text_item in bubble["texts"]:
            text_id = text_item.get("text_id", 0)
            if text_id > 0:
                translated = bubble.get("edited_text", "")
                if not translated:
                    translated = bubble.get("translated_text", "")
                id_list.append(text_id)
                print(f"  ID {text_id}: {translated}")

    print("-" * 40)

    if len(id_list) == 0:
        print(" Нет доступных ID для редактирования")
        show_page()
        return

    id_input = widgets.Text(
        value='',
        placeholder='Введите ID текста...',
        description='ID:',
        layout=widgets.Layout(width='200px')
    )

    btn_confirm = widgets.Button(
        description=" Выбрать",
        button_style='primary',
        layout=widgets.Layout(width='150px')
    )

    btn_cancel = widgets.Button(
        description=" Назад",
        button_style='danger',
        layout=widgets.Layout(width='150px')
    )

    output = widgets.Output()

    def on_confirm(b):
        if state['stop']:
            return
        try:
            text_id = int(id_input.value)
            if text_id in id_list:
                show_text_edit_interface(text_id)
            else:
                with output:
                    output.clear_output()
                    print(f" ID {text_id} не найден. Доступные: {', '.join(str(x) for x in id_list)}")
        except ValueError:
            with output:
                output.clear_output()
                print(" Введите целое число")

    def on_cancel(b):
        if state['stop']:
            return
        show_page()

    btn_confirm.on_click(on_confirm)
    btn_cancel.on_click(on_cancel)

    display(id_input, widgets.HBox([btn_confirm, btn_cancel]), output)

# =====================================================
# ФУНКЦИЯ РЕДАКТИРОВАНИЯ ТЕКСТА
# =====================================================

def show_text_edit_interface(text_id):
    """Показывает интерфейс для редактирования текста"""

    if state['stop']:
        return

    page = state['page']
    bubble, text_item = find_text_by_id(page, text_id)

    if bubble is None:
        print(f" Текст с ID {text_id} не найден")
        show_id_selection()
        return

    clear_output(wait=True)

    page_index = state['page_index']
    total_pages = state['total_pages']
    image_name = state['image_name']

    print("=" * 60)
    print(f" СТРАНИЦА {page_index + 1}/{total_pages}: {image_name} (РЕДАКТИРОВАНИЕ ID {text_id})")
    print("=" * 60)

    # Показываем картинку с боксами и ID
    img_clean_translated, fig = get_translated_image_with_boxes(state['img'], state['page'])
    state['img_clean'] = img_clean_translated
    show_images_from_fig(fig)

    translated_text = bubble.get("translated_text", "")
    current_edited = bubble.get("edited_text", "")

    display_text = current_edited if current_edited else translated_text

    print("=" * 60)
    print(f" РЕДАКТИРОВАНИЕ ТЕКСТА ID {text_id}")
    print("=" * 60)
    print(f" Текущий перевод: {display_text}")
    print("-" * 40)

    text_input = widgets.Textarea(
        value=display_text,
        placeholder='Введите новый перевод...',
        description='Новый перевод:',
        layout=widgets.Layout(width='90%', height='100px')
    )

    btn_save = widgets.Button(
        description=" Сохранить",
        button_style='success',
        layout=widgets.Layout(width='150px')
    )

    btn_cancel = widgets.Button(
        description=" Отмена",
        button_style='danger',
        layout=widgets.Layout(width='150px')
    )

    output = widgets.Output()

    def on_save(b):
        if state['stop']:
            return
        new_text = text_input.value.strip()
        if new_text:
            bubble["edited_text"] = new_text
            with output:
                output.clear_output()
                print(f" Текст ID {text_id} сохранен!")

            # Показываем обновленную страницу
            show_updated_page()
        else:
            with output:
                output.clear_output()
                print(" Текст не может быть пустым")

    def on_cancel(b):
        if state['stop']:
            return
        # Возвращаемся к выбору ID
        show_id_selection()

    btn_save.on_click(on_save)
    btn_cancel.on_click(on_cancel)

    display(text_input, widgets.HBox([btn_save, btn_cancel]), output)

# =====================================================
# ФУНКЦИЯ ПОКАЗА ОБНОВЛЕННОЙ СТРАНИЦЫ
# =====================================================

def show_updated_page():
    """Показывает обновленную страницу после редактирования"""

    if state['stop']:
        return

    clear_output(wait=True)

    page_index = state['page_index']
    total_pages = state['total_pages']
    image_name = state['image_name']

    print("=" * 60)
    print(f" СТРАНИЦА {page_index + 1}/{total_pages}: {image_name} (ОБНОВЛЕНА)")
    print("=" * 60)

    # Получаем обновленную картинку
    img_clean_translated, fig = get_translated_image_with_boxes(state['img'], state['page'])
    state['img_clean'] = img_clean_translated

    # Показываем фигуру ОДИН РАЗ
    show_images_from_fig(fig)

    # Показываем меню после редактирования
    show_edit_menu()

# =====================================================
# ФУНКЦИЯ МЕНЮ ПОСЛЕ РЕДАКТИРОВАНИЯ
# =====================================================

def show_edit_menu():
    """Показывает меню после редактирования"""

    if state['stop']:
        return

    print("\n" + "-" * 40)
    print(" Продолжить редактирование?")
    print("-" * 40)

    btn_again = widgets.Button(
        description=" Выбрать другой ID",
        button_style='primary',
        layout=widgets.Layout(width='200px')
    )

    btn_save = widgets.Button(
        description=" Сохранить и далее",
        button_style='success',
        layout=widgets.Layout(width='200px')
    )

    btn_stop = widgets.Button(
        description=" Сохранить и закончить",
        button_style='danger',
        layout=widgets.Layout(width='200px')
    )

    output = widgets.Output()

    def on_again(b):
        if state['stop']:
            return
        show_id_selection()

    def on_save(b):
        if state['stop']:
            return
        save_current_page()

        # Переходим к следующей странице
        state['page_index'] += 1
        if state['page_index'] < state['total_pages']:
            page = state['pages'][state['page_index']]
            image_name = page["image"]

            if image_name in state['image_dict']:
                img = cv2.imread(state['image_dict'][image_name])
                if img is not None:
                    state['img'] = img
                    state['page'] = page
                    state['image_name'] = image_name
                    show_page()
                    return

        # Все страницы обработаны
        save_json()
        print("\n" + "=" * 60)
        print(" ВСЕ СТРАНИЦЫ ОБРАБОТАНЫ!")
        print("=" * 60)
        state['stop'] = True

    def on_stop(b):
        if state['stop']:
            return
        save_current_page()
        save_json()
        state['stop'] = True
        clear_output(wait=True)
        print("\n" + "=" * 60)
        print(" РАБОТА ЗАВЕРШЕНА")
        print("=" * 60)
        print(f" Сохранено страниц: {state['saved_count']}")
        print("=" * 60)
        display(widgets.HTML("<i>Редактор остановлен. Закройте эту ячейку или перезапустите для новой сессии.</i>"))

    btn_again.on_click(on_again)
    btn_save.on_click(on_save)
    btn_stop.on_click(on_stop)

    display(widgets.HBox([btn_again, btn_save, btn_stop]), output)

# =====================================================
# ЗАПУСК РЕДАКТОРА
# =====================================================

total_pages = len(pages)

print("=" * 60)
print(f" Всего страниц: {total_pages}")
print("=" * 60)

while True:
    try:
        start_page = int(input(f" С какой страницы начать? (1-{total_pages}): "))
        if 1 <= start_page <= total_pages:
            break
    except:
        pass
    print(f" Введите число от 1 до {total_pages}")

start_index = start_page - 1

state['pages'] = pages
state['image_dict'] = image_dict
state['output_dir'] = output_dir
state['total_pages'] = total_pages
state['saved_count'] = 0
state['stop'] = False
state['page_index'] = start_index

page = pages[start_index]
image_name = page["image"]

if image_name in image_dict:
    img = cv2.imread(image_dict[image_name])
    if img is not None:
        state['img'] = img
        state['page'] = page
        state['image_name'] = image_name
        show_page()
    else:
        print(f" Ошибка чтения: {image_name}")
else:
    print(f" Изображение не найдено: {image_name}")


 РАБОТА ЗАВЕРШЕНА
 Сохранено страниц: 2


HTML(value='<i>Редактор остановлен. Закройте эту ячейку или перезапустите для новой сессии.</i>')